Extração e Carregamento dos Dados

Nesta etapa inicial, carregamos a base de dados da **Telecom X** que foi previamente tratada na Fase 1 do projeto.
O objetivo é garantir que estamos trabalhando com dados limpos, sem valores nulos e com as tipagens (como `Mensalidade` e `Total_Pago`) devidamente corrigidas para o processamento matemático.

In [ ]:
import pandas as pd
import numpy as np

# Carregando os dados que você acabou de colocar na pasta
df = pd.read_csv('telecom_churn_limpo.csv')

# Visualizando as primeiras linhas para garantir que os dados vieram certos
df.head()

# Seleção de Atributos e Padronização

Seleção de Atributos

Para que o modelo de Machine Learning aprenda padrões reais de comportamento, precisamos remover "ruídos".
* **Remoção do `customerID`**: Por ser um identificador único, ele não possui valor preditivo. Se mantido, o modelo poderia sofrer *overfitting*, tentando "decorar" o comportamento de clientes específicos em vez de aprender regras gerais.
* **Foco em Comportamento**: Mantemos apenas variáveis demográficas, serviços assinados e informações financeiras que influenciam diretamente a decisão de cancelamento (Churn).

In [ ]:
# 1. Removendo colunas irrelevantes para o modelo (como o ID do cliente)
# O 'customerID' é único para cada linha e não ajuda o modelo a aprender padrões.
# O 'Churn' original (texto) será descartado pois usaremos a versão binária.
colunas_para_remover = ['customerID', 'Churn']
df_relevante = df.drop(columns=colunas_para_remover)

# 2. Verificando os tipos de dados para garantir a padronização
# É importante que colunas como 'Mensalidade' e 'Total_Pago' sejam numéricas.
print("Tipos de dados após a limpeza inicial:")
print(df_relevante.dtypes)

# 3. Visualizando o novo DataFrame
print(f"\nDimensões do dataset relevante: {df_relevante.shape}")
df_relevante.head()

# Codificação de Variáveis (Encoding)

Nesta etapa, transformamos as variáveis categóricas (texto) em formatos numéricos.
Utilizamos o método **One-Hot Encoding** através da função `pd.get_dummies()`.

**Por que este método?** Diferente do Label Encoding (que atribui números 0, 1, 2...), o One-Hot Encoding cria colunas binárias para cada categoria. Isso evita que o modelo de Machine Learning interprete erroneamente que existe uma ordem de importância ou hierarquia entre as categorias (por exemplo, achar que a forma de pagamento 'Cartão de Crédito' é "maior" que 'Boleto').

In [ ]:
# 1. Aplicando o One-Hot Encoding em todo o DataFrame
# O pandas identifica automaticamente as colunas de texto (object) e as transforma
df_final = pd.get_dummies(df_relevante)

# 2. Verificando o resultado
print(f"Número de colunas antes: {df_relevante.shape[1]}")
print(f"Número de colunas após o Encoding: {df_final.shape[1]}")

# 3. Visualizando as primeiras linhas das novas colunas
df_final.head()

# Verificação da Proporção de Evasão

Antes de treinar os modelos, precisamos entender a distribuição da nossa variável alvo (`Churn_binario`).
O desbalanceamento de classes acontece quando uma categoria é muito mais frequente que a outra. Se não identificado, isso pode levar o modelo a ter uma alta acurácia "falsa", onde ele acerta muito quem permanece na empresa, mas falha justamente em detectar quem tem potencial de sair (o nosso objetivo principal).

In [ ]:
# 1. Calculando a contagem absoluta e a proporção
contagem_churn = df_final['Churn_binario'].value_counts()
proporcao_churn = df_final['Churn_binario'].value_counts(normalize=True) * 100

# 2. Exibindo os resultados de forma clara
print("Distribuição das Classes:")
print(f"Não Evasão (0): {contagem_churn[0]} clientes ({proporcao_churn[0]:.2f}%)")
print(f"Evasão (1):     {contagem_churn[1]} clientes ({proporcao_churn[1]:.2f}%)")

# 3. Visualização gráfica para o relatório
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(6, 4))
sns.countplot(x='Churn_binario', data=df_final, palette='viridis')
plt.title('Distribuição da Variável Alvo (Churn)')
plt.xlabel('Evasão (0 = Não, 1 = Sim)')
plt.ylabel('Quantidade de Clientes')
plt.show()

# Normalização e Padronização

Nesta etapa, ajustamos a escala das variáveis numéricas contínuas (como `Tenure`, `Mensalidade` e `Total_Pago`).

**Por que padronizar?**
Modelos como a **Regressão Logística** (que pretendemos testar) são sensíveis à escala dos dados. Se uma variável tem valores entre 0 e 1 (ex: Gênero) e outra entre 0 e 8000 (ex: Total Pago), o modelo pode interpretar erroneamente que a variável com valores maiores é mais importante.

Utilizaremos o **StandardScaler**, que transforma os dados para que tenham média 0 e desvio padrão 1, colocando todas as variáveis na mesma "grandeza" matemática.

In [ ]:
from sklearn.preprocessing import StandardScaler

# 1. Identificando automaticamente as colunas numéricas (que não são resultado do get_dummies)
# Vamos pegar colunas que possuem mais de 5 valores únicos (o que exclui as colunas binárias 0 e 1)
colunas_numericas = [col for col in df_final.columns if df_final[col].nunique() > 5]

print(f"Colunas identificadas para padronização: {colunas_numericas}")

# 2. Criando o padronizador
scaler = StandardScaler()

# 3. Aplicando a padronização
df_final[colunas_numericas] = scaler.fit_transform(df_final[colunas_numericas])

# 4. Verificando o resultado
print("\nDados padronizados (primeiras 5 linhas):")
df_final[colunas_numericas].head()

# Correlação e Seleção de Variáveis

Análise de Correlação

Nesta etapa, geramos uma matriz de correlação para identificar como cada variável se relaciona com a evasão (`Churn_binario`).

* **Correlação Positiva:** Quanto maior o valor da variável, maior a chance de Churn (ex: Contratos Mensais).
* **Correlação Negativa:** Quanto maior o valor, menor a chance de Churn (ex: Tempo de casa/Tenure).

Essa análise nos permite validar hipóteses levantadas na EDA e selecionar os melhores atributos para o modelo.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib.pyplot as plt

# 1. Calculando as correlações
correlacoes = df_final.corr()['Churn_binario'].sort_values(ascending=False).drop('Churn_binario')

# 2. Dicionário de tradução para os termos mais comuns
# Você pode adicionar mais termos aqui se aparecerem outros no seu gráfico
traducoes = {
    'Month-to-month': 'Mensal',
    'Fiber optic': 'Fibra Óptica',
    'Electronic check': 'Cheque Eletrônico',
    'tenure': 'Tempo de Casa',
    'MonthlyCharges': 'Mensalidade',
    'TotalCharges': 'Total Pago',
    'Two year': 'Contrato 2 Anos',
    'One year': 'Contrato 1 Ano',
    'PaperlessBilling': 'Fatura Digital',
    'No internet service': 'Sem Internet',
    'OnlineSecurity': 'Segurança Online',
    'TechSupport': 'Suporte Técnico',
    'DeviceProtection': 'Proteção de Dispositivo',
    'OnlineBackup': 'Backup Online'
}

# 3. Limpando e traduzindo os nomes
nomes_traduzidos = []
for col in correlacoes.index:
    nome = col.split('_')[-1] # Pega a parte após o '_'
    nome = traducoes.get(nome, nome) # Traduz se estiver no dicionário, senão mantém
    nomes_traduzidos.append(nome)

# 4. Criando o gráfico
plt.figure(figsize=(10, 8))
plt.barh(nomes_traduzidos, correlacoes.values, color='skyblue')
plt.title('Influência das Variáveis na Evasão (Churn)', fontsize=14)
plt.xlabel('Força da Correlação', fontsize=12)
plt.gca().invert_yaxis()
plt.grid(axis='x', linestyle='--', alpha=0.6)
plt.tight_layout()
plt.show()

# Análises Direcionadas (Comportamento de Evasão)

Nesta etapa, aprofundamos a investigação em variáveis críticas identificadas na matriz de correlação. O objetivo é visualizar a distribuição dos dados entre clientes que permaneceram e clientes que saíram.

* **Tempo de Casa (Tenure) vs Evasão**: Avaliamos se clientes mais novos têm maior propensão ao cancelamento.
* **Total Gasto vs Evasão**: Analisamos se o valor acumulado pago pelo cliente influencia sua decisão de sair.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# 1. Identificando os nomes reais das colunas no seu DataFrame
# Vamos procurar por termos parecidos para evitar o erro de 'not found'
col_tenure = [c for c in df.columns if 'tenure' in c.lower() or 'tempo' in c.lower()][0]
col_total = [c for c in df.columns if 'total' in c.lower()][0]
col_churn = 'Churn_binario'

print(f"Usando as colunas: {col_tenure}, {col_total} e {col_churn}")

# Configurando o estilo
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 6))

# 1. Gráfico de Boxplot: Tempo de Casa x Evasão
plt.subplot(1, 2, 1)
sns.boxplot(x=col_churn, y=col_tenure, data=df, palette='viridis')
plt.title('Distribuição: Tempo de Casa x Evasão', fontsize=14)
plt.xlabel('Evasão (0 = Não, 1 = Sim)')
plt.ylabel('Meses de Contrato')

# 2. Gráfico de Boxplot: Total Gasto x Evasão
plt.subplot(1, 2, 2)
sns.boxplot(x=col_churn, y=col_total, data=df, palette='magma')
plt.title('Distribuição: Total Gasto x Evasão', fontsize=14)
plt.xlabel('Evasão (0 = Não, 1 = Sim)')
plt.ylabel('Valor Total (R$)')

plt.tight_layout()
plt.show()

# Modelagem Preditiva

Para validar a eficácia do nosso modelo de Machine Learning, dividimos a base de dados em dois conjuntos distintos:

* **Conjunto de Treino (70%):** Utilizado para que o modelo aprenda os padrões e relações entre os serviços contratados e a evasão.
* **Conjunto de Teste (30%):** Funciona como um "exame final". Testamos o modelo com dados que ele nunca viu para medir sua capacidade real de prever o Churn em novos clientes.

Essa separação é fundamental para evitar o **overfitting** (quando o modelo apenas decora os dados de treino e falha ao enfrentar situações reais).

In [ ]:
from sklearn.model_selection import train_test_split

# 1. Definindo as variáveis explicativas (X) e o alvo (y)
# X contém todos os atributos (serviços, valores, etc)
# y contém apenas a resposta (Churn_binario)
X = df_final.drop(columns=['Churn_binario'])
y = df_final['Churn_binario']

# 2. Realizando a separação com 30% para teste
# O random_state=42 garante que, se rodarmos o código de novo, a divisão será a mesma
# O stratify=y mantém a mesma proporção de Churn em ambas as bases
X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y,
    test_size=0.30,
    random_state=42,
    stratify=y
)

# 3. Exibindo o balanço final das bases em português
print(f"Base de Treino: {len(X_treino)} registros")
print(f"Base de Teste:  {len(X_teste)} registros")
print("\nVerificação de consistência (Proporção de Evasão no Treino):")
print(y_treino.value_counts(normalize=True).map('{:.2%}'.format))

# Tratando Nulos Remanescentes

In [ ]:
# 1. Verificando onde estão os nulos teimosos
print("Valores nulos por coluna no X_treino:")
print(X_treino.isnull().sum().sum())

# 2. Preenchendo valores nulos com zero (estratégia comum para Total Pago de novos clientes)
# Ou removendo as linhas, caso prefira: X_treino = X_treino.dropna()
X_treino = X_treino.fillna(0)
X_teste = X_teste.fillna(0)

print("\n✅ Nulos tratados! Agora as bases estão prontas.")

# Criação e Treinamento dos Modelos

Para este desafio, selecionamos dois modelos distintos para comparar desempenhos:

1. **Regressão Logística**:
   - **Justificativa**: É um modelo linear simples, rápido e altamente interpretável. Ele funciona muito bem para problemas de classificação binária (Sim/Não).
   - **Normalização**: Exige dados padronizados (média 0 e desvio padrão 1) para garantir que variáveis com valores grandes (como 'Total Pago') não dominem as variáveis menores durante o cálculo dos pesos.

2. **Random Forest (Floresta Aleatória)**:
   - **Justificativa**: É um modelo de conjunto (ensemble) que combina várias árvores de decisão para reduzir o erro. É muito robusto contra ruídos nos dados.
   - **Normalização**: Não exige normalização, pois as decisões de "quebra" dos dados nas árvores não dependem da escala das variáveis.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# --- MODELO 1: Regressão Logística ---
# Criando o modelo
modelo_logistico = LogisticRegression(random_state=42)

# Treinando o modelo com a base de treino
modelo_logistico.fit(X_treino, y_treino)

# Realizando as previsões
previsoes_logistica = modelo_logistico.predict(X_teste)


# --- MODELO 2: Random Forest ---
# Criando o modelo com 100 árvores de decisão
modelo_floresta = RandomForestClassifier(n_estimators=100, random_state=42)

# Treinando o modelo
modelo_floresta.fit(X_treino, y_treino)

# Realizando as previsões
previsoes_floresta = modelo_floresta.predict(X_teste)

print("✅ Modelos treinados e previsões realizadas com sucesso!")

# Avaliação dos Modelos

Nesta etapa, utilizamos diversas métricas para entender o desempenho real dos nossos modelos:

* **Acurácia**: Indica o percentual total de acertos.
* **Precisão**: De todos os que o modelo disse que iam sair, quantos realmente saíram?
* **Recall (Sensibilidade)**: De todos os que realmente saíram, quantos o modelo conseguiu detectar? (Vital para estratégias de retenção).
* **F1-Score**: O equilíbrio entre Precisão e Recall.
* **Matriz de Confusão**: Mostra visualmente os acertos e erros (Falsos Positivos e Falsos Negativos).

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt

def avaliar_modelo(y_real, y_pred, nome_modelo):
    print(f"--- Relatório de Desempenho: {nome_modelo} ---")
    # Traduzindo os termos do relatório para Português
    print(classification_report(y_real, y_pred, target_names=['Permanência', 'Evasão']))

    # Criando a Matriz de Confusão
    fig, ax = plt.subplots(figsize=(6, 4))
    cm = confusion_matrix(y_real, y_pred)
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['Permanência', 'Evasão'])
    disp.plot(ax=ax, cmap='Blues', values_format='d')
    plt.title(f'Matriz de Confusão: {nome_modelo}')
    plt.xlabel('Previsão do Modelo')
    plt.ylabel('Valor Real (Empresa)')
    plt.show()
    print("\n")

# Avaliando os dois modelos
avaliar_modelo(y_teste, previsoes_logistica, "Regressão Logística")
avaliar_modelo(y_teste, previsoes_floresta, "Random Forest")

# Análise de Importância das Variáveis

Para entender como os modelos tomam suas decisões, analisamos o peso que cada variável tem na previsão final. Isso permite que a empresa saiba exatamente onde agir para evitar o Churn.

* **Regressão Logística**: Analisamos os **coeficientes**. Valores positivos altos indicam que a variável aumenta a chance de evasão; valores negativos indicam que ela ajuda a reter o cliente.
* **Random Forest**: Analisamos a **importância relativa** baseada na redução da impureza (Gini). Ela nos diz quais variáveis foram mais decisivas para as divisões das árvores de decisão.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Dicionário de tradução que usamos antes (ajustado para os nomes das colunas do modelo)
traducoes = {
    'tenure': 'Tempo de Casa',
    'MonthlyCharges': 'Mensalidade',
    'TotalCharges': 'Total Pago',
    'Contract_Month-to-month': 'Contrato Mensal',
    'Contract_One year': 'Contrato 1 Ano',
    'Contract_Two year': 'Contrato 2 Anos',
    'InternetService_Fiber optic': 'Fibra Óptica',
    'PaymentMethod_Electronic check': 'Cheque Eletrônico',
    'PaperlessBilling_Yes': 'Fatura Digital'
}

def limpar_nome(nome):
    for eng, pt in traducoes.items():
        if eng in nome:
            return pt
    return nome.split('_')[-1]

# 1. Importância na Regressão Logística (Coeficientes)
coeficientes = pd.DataFrame({
    'Variável': [limpar_nome(col) for col in X.columns],
    'Peso': modelo_logistico.coef_[0]
}).sort_values(by='Peso', ascending=False)

# 2. Importância no Random Forest
importancias_rf = pd.DataFrame({
    'Variável': [limpar_nome(col) for col in X.columns],
    'Importância': modelo_floresta.feature_importances_
}).sort_values(by='Importância', ascending=False)

# --- Visualização ---
plt.figure(figsize=(16, 6))

# Gráfico da Regressão Logística
plt.subplot(1, 2, 1)
sns.barplot(x='Peso', y='Variável', data=coeficientes.head(10), palette='coolwarm')
plt.title('Top 10 Fatores de Influência (Regressão Logística)', fontsize=12)
plt.xlabel('Peso (Coeficiente)')

# Gráfico do Random Forest
plt.subplot(1, 2, 2)
sns.barplot(x='Importância', y='Variável', data=importancias_rf.head(10), palette='viridis')
plt.title('Top 10 Variáveis Mais Decisivas (Random Forest)', fontsize=12)
plt.xlabel('Importância Relativa')

plt.tight_layout()
plt.show()

--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------

# Relatório Final e Conclusões Estratégicas

## 🏁 12. Conclusão e Recomendações de Negócio

Este projeto teve como objetivo prever a evasão de clientes (Churn) da **Telecom X** e identificar os fatores que levam ao cancelamento. Após o processamento dos dados e o treinamento de modelos de Machine Learning, chegamos às seguintes conclusões:

### 📊 1. Desempenho dos Modelos
Comparamos a **Regressão Logística** e o **Random Forest**.
* O modelo de **Regressão Logística** mostrou-se mais equilibrado para o negócio, apresentando um bom **Recall (Sensibilidade)**. Isso significa que ele é eficaz em identificar a maioria dos clientes que pretendem sair, permitindo uma ação preventiva rápida.
* O **Random Forest** apresentou ótima acurácia, mas tendeu a dar mais peso a variáveis financeiras como o 'Total Pago'.

### 🔍 2. Principais Fatores de Influência (Insights)
Com base na análise de importância das variáveis, os três principais "vilões" do Churn são:
1.  **Tipo de Contrato**: Clientes com contrato **Mensal** têm uma propensão drasticamente maior a cancelar do que aqueles em planos anuais.
2.  **Tempo de Casa (Tenure)**: O risco de evasão é altíssimo nos **primeiros 6 meses**. Se o cliente ultrapassa o primeiro ano, a chance de ele permanecer fiel aumenta consideravelmente.
3.  **Tecnologia de Internet**: Clientes de **Fibra Óptica** estão saindo mais que os de DSL, o que pode indicar problemas de preço ou instabilidade técnica nessa tecnologia específica.

### 💡 3. Estratégias de Retenção Propostas
Para reduzir a taxa de evasão, recomendamos as seguintes ações:

* **Incentivo à Fidelização**: Criar campanhas de desconto para migrar clientes do plano "Mensal" para o "Anual". O custo do desconto é menor que o custo de perder o cliente.
* **Onboarding Especializado**: Focar o suporte técnico e o atendimento ao cliente nos primeiros 3 meses de contrato, garantindo que o cliente tenha uma experiência perfeita no início da jornada.
* **Revisão da Oferta de Fibra**: Investigar a satisfação dos usuários de Fibra Óptica para entender se a evasão é motivada por valor ou qualidade do sinal.
* **Fatura Digital**: Incentivar o uso de débito automático e faturas digitais, que mostraram correlação com uma maior retenção.